In [1]:
# ════════════════════════════════════════════════════════════════════════════════════════════
# ScoutIA — TEST : MOINS D'IMAGES POUR Q2 « QUI ENGAGE » (attribution en résolution NORMALE) — 11/11/2026
# Q2 est la question la plus chère (jusqu'à ~106 images à 1 image/s). Un seul paramètre change : l'ÉCART ENTRE DEUX
# IMAGES de Q2. Tout le reste est identique au test du 09/11 (43 vrais buts, heures de la vérité terrain, sans rejouer le
# pipeline, mêmes questions, résolution normale).
# RÉFÉRENCE = réponses en résolution normale (1 image/s) du run du 11/11 (41 justes, 0 faux, 2 non attribués), RELUES
# depuis attribution_restest_<match>.json
# (gratuites si ces fichiers sont en entrée). Q1, Q3 et les tenues sont les mêmes questions sur les mêmes images : relues
# aussi quand elles existent ; seules les nouvelles Q2 (et les Q3 devenues nécessaires) sont payées.
# CRITÈRES FIXÉS AVANT (chaque variante comparée à la référence, sur les 43 buts) :
#   1. aucune attribution FAUSSE (la référence en a 0) ;
#   2. au plus 1 but attribué juste de moins que la référence (run du 11/11 : 41 → au moins 40) ;
# On retient la variante avec le PLUS GRAND écart qui passe les deux critères.
# Limites : (a) en production, Q2 démarre au coup d'envoi validé (46 images à 1/s), ici 15 s après le but (fenêtre plus
# longue, faute de coup d'envoi dans la vérité terrain) ; (b) une différence peut aussi venir de la variabilité de Gemini.
# Coût prévu : ≈ 2 $ (2 s) + ≈ 1,4 $ (3 s) si la référence est relue ; ≈ +5,4 $ si elle doit être repayée.
# Entrées : codeprojectai, les 10 vidéos, mziparameter, test_resolution_gemini_cellule.py, ET la sortie du test du 09/11
# (attribution_restest_<match>.json).
# ════════════════════════════════════════════════════════════════════════════════════════════
VARIANTES = {"pas2": 2, "pas3": 3}   # nom → secondes entre deux images de Q2 (référence = 1 s, relue)
EXIGER_REFERENCE = True              # sans effet depuis le 11/11 : la référence à 1 image/s n'est JAMAIS repayée
# 11/11 : à 1 image/s, Q2 dépasse souvent la limite de 30 Mo par requête de Gemini (« Request body too large ») → une
# référence repayée aujourd'hui serait CASSÉE. Si les réponses valides (run 41/0/2) ne sont pas en entrée, la référence
# est la valeur MESURÉE de ce run, fixée ici, et les variantes sont jugées sur la vérité terrain :
REF_FIXE = {"justes": 41, "faux": 0, "non_attribues": 2}
MATCHS_A_TESTER = ["spa", "minerois_ster", "franchimont", "raeren", "andrimont", "p1_minerois", "goe", "stembert",
                   "wanze", "juprelle"]
PLAFOND_TOTAL = 8.0                  # $ (sécurité)

import os, glob, json, re, collections, subprocess, sys
try:
    import cv2
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless"], check=False); import cv2
import numpy as np
from google import genai
from google.genai import types

# ── vérité terrain et vidéos, reprises du notebook de test (même source que tous les tests) ──
_src = next(iter(sorted(glob.glob("/kaggle/input/**/test_resolution_gemini_cellule.py", recursive=True))
                 + glob.glob("/kaggle/working/test_resolution_gemini_cellule.py")), None)
if not _src:
    raise SystemExit("❌ test_resolution_gemini_cellule.py introuvable (vérité terrain)")
_txt = open(_src, encoding="utf-8").read()


def _bloc(nom):
    i = _txt.index(nom + " = {"); j = _txt.index("{", i); n = 0
    for k in range(j, len(_txt)):
        n += {"{": 1, "}": -1}.get(_txt[k], 0)
        if n == 0:
            return _txt[i:k + 1]


_ns = {}
exec(_txt[_txt.index("V_ = "):_txt.index("\n", _txt.index("V_ = "))], _ns)
for _n in ("MATCHS", "VERITE_EQUIPES", "VRAIES_TENUES_TOUTES"):
    exec(_bloc(_n), _ns)
MATCHS, VERITE, VRAIES_TENUES = _ns["MATCHS"], _ns["VERITE_EQUIPES"], _ns["VRAIES_TENUES_TOUTES"]

# ── clé Gemini : param.env du dataset privé « mziparameter » (comme les autres notebooks), sinon secrets Kaggle ──
_pe = next(iter(glob.glob("/kaggle/input/**/mziparameter/**/param.env", recursive=True)
                + glob.glob("/kaggle/input/**/param.env", recursive=True)), None)
if _pe:
    for _l in open(_pe, encoding="utf-8"):
        _l = _l.strip()
        if _l and not _l.startswith("#") and "=" in _l:
            _k, _v = _l.split("=", 1)
            if _k.strip() == "GEMINI_API_KEY" and not os.environ.get("GEMINI_API_KEY"):
                os.environ["GEMINI_API_KEY"] = _v.strip().strip('"').strip("'")
if not os.environ.get("GEMINI_API_KEY"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY") or ""
    except Exception:
        pass
if not os.environ.get("GEMINI_API_KEY"):
    raise SystemExit("❌ GEMINI_API_KEY introuvable (ajouter « mziparameter » en entrée)")
CLIENT = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
JET = collections.defaultdict(lambda: [0, 0, 0])          # par variante : questions, jetons entrée, sortie+réflexion
cout = lambda r: JET[r][1] / 1e6 * 0.75 + JET[r][2] / 1e6 * 3.75
fmt = lambda s: f"{int(s)//60:02d}:{int(s)%60:02d}"
sec = lambda h: int(h.split(":")[0]) * 60 + float(h.split(":")[1])

QTENUES = """These 16 images come from an amateur football match video (a camera following the play), taken at different moments.
Identify the OUTFIELD players' kit of each of the two teams (ignore goalkeepers and the referee, describe them separately).
For each team give the main colour(s) of the shirt, the shorts and the socks, in simple English colour words
(e.g. "yellow", "red and white", "dark blue", "light blue", "black"). Also say whether the two teams' shirts share a colour.
Answer ONLY with JSON:
{"team_1": {"shirt": "...", "shorts": "...", "socks": "..."}, "team_2": {"shirt": "...", "shorts": "...", "socks": "..."},
 "goalkeepers": "...", "referee": "...", "shirts_share_a_colour": true|false, "certainty": "high|medium|low"}"""
# Questions identiques à l'attribution validée (notebook de test) : exécution des 4 définitions telles quelles
_q = {}
_i = _txt.index("_AENT = "); _j = _txt.index("\n\n\ndef _aeq")
exec(_txt[_i:_j], _q)
AENT, AQ1, AQ2, AQ3 = _q["_AENT"], _q["_AQ1"], _q["_AQ2"], _q["_AQ3"]
AUTRE = {"A": "B", "B": "A"}


def rappel(a0, a1, b0, b1, toujours=False):
    mots = lambda x: set(x.lower().replace(",", " ").split()) - {"and", "dark", "light", "with"}
    com = mots(a0) & mots(b0)
    if com or toujours:
        t = f" Both teams have {' and '.join(sorted(com))} on their shirts." if com else " Shirts alone can be misleading (shadows, light)."
        return t + f" To tell the teams apart, look at the SHORTS: team A has {a1} shorts, team B has {b1} shorts."
    return ""


def demander(rep, fichier, cle, res, cap, instants, consigne, etiquette=True):
    k = f"{res}|{cle}"
    if k in rep:
        return rep[k]
    if sum(cout(x) for x in list(JET)) > PLAFOND_TOTAL:
        raise SystemExit(f"❌ plafond {PLAFOND_TOTAL} $ atteint")
    ims = []
    for i, t in enumerate(instants, 1):
        cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
        ok, img = cap.read()
        img = img.copy() if ok else np.zeros((1080, 1920, 3), np.uint8)
        if etiquette:
            cv2.putText(img, f"{i}  {fmt(t)}", (14, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.8, (0, 255, 255), 4)
        ims.append(types.Part.from_bytes(data=cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 85])[1].tobytes(),
                                         mime_type="image/jpeg"))
    cfg = {"temperature": 0.0, "thinking_config": {"thinking_level": "low"}}
    if res == "reduite":
        cfg["media_resolution"] = f"MEDIA_RESOLUTION_{RESOLUTION_REDUITE.upper()}"
    try:
        r = CLIENT.models.generate_content(model="gemini-3.8-flash", contents=[consigne] + ims, config=cfg)
        um = getattr(r, "usage_metadata", None)
        JET[res][0] += 1; JET[res][1] += int(getattr(um, "prompt_token_count", 0) or 0)
        JET[res][2] += int(getattr(um, "candidates_token_count", 0) or 0) + int(getattr(um, "thoughts_token_count", 0) or 0)
        t_ = r.text or ""
        d, _ = json.JSONDecoder().raw_decode(t_[t_.index("{"):])
    except Exception as e:
        print(f"   ⚠️ {k} : {e}"); ERREURS[res] += 1; return None
    rep[k] = d
    json.dump(rep, open(fichier, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    return d


def eq(d, champ):
    if not d or str(d.get("certainty", "")).lower() == "low":
        return None
    v = str(d.get(champ, "unknown")).upper()
    return v if v in AUTRE else None




mots = lambda x: set(str(x).lower().replace(",", " ").replace("/", " ").split()) - {"and", "with", "dark", "light"}
score_t = lambda det, vrai: len(mots(det[0]) & mots(vrai[0])) * 2 + len(mots(det[1]) & mots(vrai[1]))

# ── contrôle AVANT paiement : les réponses de référence du 09/11 sont-elles là ? ──
_REP = {}
_manque = []
for M in MATCHS_A_TESTER:
    r = {}
    for p in sorted(glob.glob(f"/kaggle/input/**/attribution_restest_{M}.json", recursive=True)) + \
             [f"/kaggle/working/attribution_restest_{M}.json"]:
        if os.path.exists(p):
            r.update(json.load(open(p, encoding="utf-8")))
    _REP[M] = r
    n_ref = sum(1 for k in r if k.startswith("normale|") and k.endswith("|Q2"))
    print(f"   {M:14s} : {n_ref} Q2 de référence relues / {len(VERITE.get(M, {}))} buts")
    if n_ref < len(VERITE.get(M, {})):
        _manque.append(M)
ERREURS = collections.Counter()
REF_RELUE = not _manque
if _manque:
    print(f"   ⚠️ référence absente ou incomplète pour {_manque} : elle n'est PAS repayée (limite 30 Mo) — "
          f"référence fixe {REF_FIXE}")
if False:
    raise SystemExit(f"❌ réponses de référence du 09/11 absentes pour {_manque} : ajouter la sortie du test du 09/11 "
                     f"(attribution_restest_<match>.json) en entrée, ou mettre EXIGER_REFERENCE = False (≈ +5,4 $)")

LIBS = (["ref"] if REF_RELUE else []) + list(VARIANTES)
BILAN = {l: collections.Counter() for l in LIBS}
DETAIL = []
N_IMG_Q2 = collections.Counter()
for M in MATCHS_A_TESTER:
    video = MATCHS[M]["video"]
    if not os.path.exists(video):
        print(f"⚠️ {M} : vidéo absente ({video}) — match ignoré"); continue
    fichier = f"/kaggle/working/attribution_restest_{M}.json"
    rep = _REP[M]
    cap = cv2.VideoCapture(video)
    ko = float(MATCHS[M].get("coup_envoi_s") or 0)
    deb = max(ko + 60, 15 * 60); fin = deb + 30 * 60
    buts = sorted(VERITE[M], key=sec)
    print(f"\n{'#' * 90}\n# {M} — {len(buts)} vrais buts\n{'#' * 90}")
    dt = demander(rep, fichier, f"{M}|TENUES", "normale", cap, [deb + i * (fin - deb) / 15 for i in range(16)], QTENUES, etiquette=False)
    if not dt:
        print("   tenues non détectées"); cap.release(); continue
    T1, T2 = dt.get("team_1", {}), dt.get("team_2", {})
    TD = ((str(T1.get("shirt")), str(T1.get("shorts"))), (str(T2.get("shirt")), str(T2.get("shorts"))))
    d_, i_ = score_t(TD[0], VRAIES_TENUES[M][0]) + score_t(TD[1], VRAIES_TENUES[M][1]), \
             score_t(TD[0], VRAIES_TENUES[M][1]) + score_t(TD[1], VRAIES_TENUES[M][0])
    carte = None if d_ == i_ else ({"A": "A", "B": "B"} if d_ > i_ else {"A": "B", "B": "A"})
    (a0, a1), (b0, b1) = TD
    ff = dict(a0=a0, a1=a1, b0=b0, b1=b1, r=rappel(a0, a1, b0, b1))
    for i, h in enumerate(buts):
        g = sec(h); suiv = sec(buts[i + 1]) if i + 1 < len(buts) else None
        lim = (suiv - 5) if suiv else 10 ** 9
        cle = f"{M}|{h}"
        if not REF_RELUE:
            N_IMG_Q2["ref"] += len(range(int(g) + 15, int(min(g + 120, lim)) + 1))
        d1 = demander(rep, fichier, cle + "|Q1", "normale", cap, [g + d for d in range(-8, 7, 2)], AQ1.format(**ff))
        for lib in LIBS:
            pas = 1 if lib == "ref" else VARIANTES[lib]
            inst2 = list(range(int(g) + 15, int(min(g + 120, lim)) + 1, pas))
            N_IMG_Q2[lib] += len(inst2)
            d2 = demander(rep, fichier, cle + "|Q2", "normale" if lib == "ref" else lib, cap, inst2, AQ2.format(**ff))
            s1 = eq(d1, "scoring_team"); s2 = AUTRE.get(eq(d2, "kickoff_team"))
            pts = collections.Counter()
            if s1: pts[s1] += 2
            if s2: pts[s2] += 2
            d3 = None
            if not (s1 and s2 and s1 == s2):
                d3 = demander(rep, fichier, cle + "|Q3", "normale", cap,
                              [g + 1 + d for d in range(-8, 7, 2)] + list(range(int(g) + 10, int(min(g + 121, lim)), 2)),
                              AQ3.format(**dict(ff, r=rappel(a0, a1, b0, b1, toujours=True))))
                a3, k3 = eq(d3, "scoring_team"), eq(d3, "kickoff_team")
                if a3: pts[a3] += 1
                if k3: pts[AUTRE[k3]] += 1
            tete = "A" if pts["A"] > pts["B"] else "B" if pts["B"] > pts["A"] else None
            sout = 0
            if tete:
                sout = (s1 == tete) + (s2 == tete) + (bool(d3) and (eq(d3, "scoring_team") == tete or AUTRE.get(eq(d3, "kickoff_team")) == tete))
            ecart = abs(pts["A"] - pts["B"])
            if tete and s1 == tete and s2 == tete and ecart >= 4:
                niv = "confirmé"
            elif tete and ecart >= 2 and sout >= 2:
                niv = "probable"
            else:
                niv, tete = "non attribué", None
            vrai = carte.get(tete) if (carte and tete) else None
            verdict = "—" if not tete else ("?" if not carte else ("juste" if vrai == VERITE[M][h] else "FAUX"))
            BILAN[lib][(niv, verdict)] += 1
            DETAIL.append((M, h, lib, niv, verdict))
            print(f"   [{lib}] but {h} ({VERITE[M][h]}) : {niv} → {verdict} ({len(inst2)} images Q2)")
    cap.release()
    print("   coût cumulé : " + " | ".join(f"{l} {cout(l):.2f} $ ({JET[l][0]} q.)" for l in list(JET)))

# ── BILAN ──
print(f"\n{'═' * 90}\nBILAN — moins d'images pour Q2 (résolution normale)\n{'═' * 90}")
print("   (réf = réponses du 09/11 relues ; « normale » dans les coûts = Q1/Q3/tenues payées faute de réponse en cache)")
res_l = {}
for lib in LIBS:
    b = BILAN[lib]
    att = b[("confirmé", "juste")] + b[("probable", "juste")]
    faux = sum(v for k, v in b.items() if isinstance(k, tuple) and k[1] == "FAUX")
    na = sum(v for k, v in b.items() if isinstance(k, tuple) and k[0] == "non attribué")
    res_l[lib] = (att, faux)
    print(f"   {lib:5s} : {N_IMG_Q2[lib]:5d} images Q2 | justes {att} (confirmés {b[('confirmé', 'juste')]}, probables "
          f"{b[('probable', 'juste')]}) | FAUX {faux} | non attribués {na}"
          + (f" | coût payé {cout(lib):.2f} $" if lib != "ref" else ""))
print("\n   buts dont le résultat change par rapport à la référence :")
_par = {}
for m, h, lib, niv, v in DETAIL:
    _par.setdefault((m, h), {})[lib] = (niv, v)
for (m, h), d in (_par.items() if REF_RELUE else []):
    if any(d.get(l) != d.get("ref") for l in VARIANTES):
        print(f"     {m} {h} : réf {d.get('ref')} | " + " | ".join(f"{l} {d.get(l)}" for l in VARIANTES))
att_ref = res_l["ref"][0] if REF_RELUE else REF_FIXE["justes"]
print(f"\n   questions REFUSÉES ou en erreur : {dict(ERREURS) or 'aucune'}")
if sum(ERREURS.values()):
    print("   ⚠️ des questions ont échoué (Q2 des variantes, ou Q1/Q3/tenues communes) : le résultat N'EST PAS VALABLE "
          "tant que ces questions ne sont pas reposées (relancer avec la sortie de ce run en entrée : rien n'est repayé)")
retenue = None
for lib, pas in sorted(VARIANTES.items(), key=lambda x: x[1]):
    att, faux = res_l[lib]
    ok = faux == 0 and att >= att_ref - 1
    print(f"   CRITÈRES {lib} ({pas} s) : FAUX {faux} = 0 {'✅' if faux == 0 else '❌'} | justes {att} ≥ {att_ref} − 1 "
          f"{'✅' if att >= att_ref - 1 else '❌'} → {'RÉUSSI' if ok else 'ÉCHEC'}")
    if ok:
        retenue = (lib, pas)
print(f"\n   → {'à retenir : 1 image toutes les ' + str(retenue[1]) + ' s (SCOUTIA_ATTRIBUTION_PAS_Q2=' + str(retenue[1]) + ')' if retenue else 'aucune variante ne passe : garder 1 image/s'}")
print(f"   images Q2 en moins : " + " | ".join(f"{l} −{100 * (1 - N_IMG_Q2[l] / max(1, N_IMG_Q2['ref'])):.0f} %" for l in VARIANTES))

   spa            : 7 Q2 de référence relues / 7 buts
   minerois_ster  : 0 Q2 de référence relues / 5 buts
   franchimont    : 0 Q2 de référence relues / 5 buts
   raeren         : 1 Q2 de référence relues / 9 buts
   andrimont      : 0 Q2 de référence relues / 6 buts
   p1_minerois    : 0 Q2 de référence relues / 3 buts
   goe            : 0 Q2 de référence relues / 2 buts
   stembert       : 0 Q2 de référence relues / 3 buts
   wanze          : 0 Q2 de référence relues / 2 buts
   juprelle       : 0 Q2 de référence relues / 1 buts
   ⚠️ référence absente ou incomplète pour ['minerois_ster', 'franchimont', 'raeren', 'andrimont', 'p1_minerois', 'goe', 'stembert', 'wanze', 'juprelle'] : elle n'est PAS repayée (limite 30 Mo) — référence fixe {'justes': 41, 'faux': 0, 'non_attribues': 2}

##########################################################################################
# spa — 7 vrais buts
##########################################################################################